# Camada Silver

## Imports e criação de variáveis

In [0]:
# imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, upper, initcap, coalesce

# criação da sessão do Spark
spark = SparkSession.builder.appName("bronze_to_silver").getOrCreate()

In [0]:
# paths dos schemas
bronze_path = "workspace.bronze"
silver_path = "workspace.silver"

## Tabela: dm.canais

Para a tabela de dm.canais, é preciso garantir que os tipos de dados bem como os nomes das colunas estejam corretos conforme o grupo definiu.

Schema final da tabela:

```
root
 |-- Nome_Canal: string (nullable = false)
 |-- Status_Canal: string (nullable = false)
```

In [0]:
canais = spark.read.table(f"{bronze_path}.canais")

In [0]:
# checando o schema atual dos dados da bronze layer
canais.printSchema()

root
 |-- nome_canal: string (nullable = true)
 |-- canal_status: string (nullable = true)



In [0]:
canais.display() # checando os dados

nome_canal,canal_status
URA,ativo
ATENDIMENTO INICIAL,ativo
ATENDIMENTO ESPECIALIZADO,invativo
CHATBOT,ativo
WEB,invativo
email,inativo


De acordo com o observado acima, a coluna ```nome_canal``` possui uma despadronização quanto à forma que as palavras estão escritas. Para padronizar, vamos capitalizar todas elas e caso tenha alguma ocorrência futura que seja nula, vamos colocar o valor de ```Desconhecido```.

Já na coluna de ```canal_status```, existe uma padronização quanto às diferentes escritas de inativo (sejam corretas no vocabulário português ou não). Para padronizar, vamos checar a primeira letra de cada ocorrência e atribuir à um valor constante, que será ```Ativo```, ```Inativo``` ou ```Desconhecido```, caso o valor da coluna seja nulo.

In [0]:
canais = (
    canais
    # nome_canal
    .withColumn("nome_canal", 
                # se o nome do canal for nulo, substitui por desconhecido
                coalesce(initcap(col("nome_canal")), lit("Desconhecido"))
    )
    .withColumnRenamed("nome_canal", "Nome_Canal")
    .withColumn("Nome_Canal", col("Nome_Canal").cast(StringType()))
    
    # canal_status
    .withColumn("canal_status", 
                when(upper(col("canal_status")).startswith("A"), "Ativo")
                .when(upper(col("canal_status")).startswith("I"), "Inativo")
                .otherwise("Desconhecido")
    )
    .withColumnRenamed("canal_status", "Status_Canal")
)

In [0]:
# checando alterações pós tratamento
canais.display()

Nome_Canal,Status_Canal
Ura,Ativo
Atendimento Inicial,Ativo
Atendimento Especializado,Inativo
Chatbot,Ativo
Web,Inativo
Email,Inativo


In [0]:
# checando o schema atual dos dados da bronze layer
canais.printSchema()

root
 |-- Nome_Canal: string (nullable = false)
 |-- Status_Canal: string (nullable = false)



Todas as mudanças foram efetivas e deixou a coluna padronizada para o futuro.

Como existem somente 6 ocorrências dos dados, não é possível criar futuras Views somente com esta tabela, somente em conjunto de outras tabelas.

Com isso, resta partir para o salvamento da tabela na Silver Layer.

In [0]:
canais.write.format("delta").mode("overwrite").saveAsTable(f"{silver_path}.dim_canais")